# audit_trail_services_3 — Bronze → Silver (merged, with Iceberg save)

Merges the earlier iteration/dedup notebooks into a single line from bronze to silver `processed/`, **and folds in the Iceberg publish step** (previously a separate notebook). Unit / invariant tests now live in the companion notebook `audit_trail_services_3_tests_v3.ipynb`.

**Why it is organized this way**

Bronze has three problems: `\N` (the DB export uses `\N` for null), nulls, and whole-row 2x duplication — at large scale (~873M rows). Processing order:

```
Load bronze
  -> clean (api_type / timestamps / drop test IDs / zero-pad / 3x mapping join / metadata / rename)
  -> clean \N -> null              <- last step before writing temp, so cleaned values flow into temp/processed
  -> write temp                    <- cleaning checkpoint (not yet deduped)
  -> reload + split by operation + per-op dedup (+ V2 per-date)
  -> write processed/<table>_v2/<op>/  <- deduped silver (versioned path, does not overwrite old)
  -> verify + save to Iceberg
```

**Dedup strategy (large scale):** a global `dropDuplicates()` over 873M rows triggers a huge shuffle and blows up executor `/tmp`. Instead, write temp without dedup first, then dedup per `operation` (each shuffle is small); `V2_ACCOUNT_CHANGE_AGE` (~370M rows) is further chunked by `date`.

**`\N` cleaned early:** older dedup3/dedup4 only discovered `\N` after writing processed, forcing a staging round-trip (Spark can't read and write the same path at once). Here `\N`->null is moved before writing temp, so the staging dance is unnecessary.

## ⚙️ Running as a notebook job (papermill, top-to-bottom in one pass)

This notebook is designed to run unattended as a SageMaker notebook job. Notes:

- **⏱ Very long runtime:** the §4b V2 per-date dedup (~1033 dates) alone takes **~37 hours** (measured under papermill). **Set the job max runtime to >= 48h**, otherwise it gets killed on timeout before finishing — the most common reason it "doesn't finish".
- **Idempotent:** every write is `mode("overwrite")`. A rerun cleanly overwrites `temp/` and `processed/audit_trail_services_3_v2/`, so it is safe to retry.
- **Fail-fast (data-quality gate):** the §5.2 `\N` residual check `raise`s on dirty data, and §6 fails the job if the Iceberg row-count / dedup assertions don't hold — dirty data is never passed through silently. The unit tests are in the companion tests notebook.
- **Diagnostics off by default:** the §2 `\N` check and §5.3 bronze/silver null comparison are optional full-table scans (873M rows each). Job mode defaults to `RUN_DIAGNOSTICS = False`; set `True` to see them.
- **No restart-point assumption:** the job runs §0 through §6; writing temp in §3k then reloading it in §4 is an intentional checkpoint (it cuts the long cleaning lineage so the per-op writes don't recompute the joins every time).

# 0. Setup

In [ ]:
%%pyspark project.spark.compatibility
import boto3
import time

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F

In [ ]:
%%pyspark project.spark.compatibility
spark = (
    SparkSession.builder
    .appName("audit_trail_services_3")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "100")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)


In [ ]:
%%pyspark project.spark.compatibility
spark.conf.set("spark.sql.session.timeZone", "America/Toronto")


# 1. Configuration

All paths / test-ID lists / constants / target schema live here; the cells below add no further config.

In [ ]:
%%pyspark project.spark.compatibility
# ===== Paths =====
BRONZE_PARQUET_PATH = "s3://enstream-lake-silver-dev/temp_raw/audit_trail_services_3/"

# Mapping sources
# names    <- partner_configuration.csv (spec §6.1.2): merchantName carries both
#   partner_name (rows where service_provider_id == 'default') and
#   service_provider_name (rows with a specific service_provider_id)
# industry <- service_providers.csv (partner_configuration has no industry column)
PARTNER_CONFIG_CSV_PATH    = "s3://enstream-lake-bronze-dev/raw/partner_configuration.csv"
SERVICE_PROVIDERS_CSV_PATH = "s3://enstream-lake-bronze-dev/raw/service_providers.csv"   # industry only

SILVER_TEMP_PATH        = "s3://enstream-lake-silver-dev/temp/audit_trail_services_3/"
SILVER_PROCESSED_BUCKET = "enstream-lake-silver-dev"
SILVER_PROCESSED_PREFIX = "processed"
SILVER_TABLE_PREFIX     = "audit_trail_services_3"

# ===== processed/ output version (does not overwrite old paths) =====
# Old data is in processed/audit_trail_services_3/; this run writes to a versioned sibling
PROCESSED_VERSION = "v2"   # bump on the next rerun (v3, v4, ...)

# ===== Job-mode switch =====
RUN_DIAGNOSTICS = False  # True runs the optional §2/§5.3 full-table scans (873M rows each)

# processed/ root (the 29 ops + V2 sub-paths all live under this prefix)
OUT_BASE = f"s3://{SILVER_PROCESSED_BUCKET}/{SILVER_PROCESSED_PREFIX}/{SILVER_TABLE_PREFIX}_{PROCESSED_VERSION}/"
print(f"processed output root: {OUT_BASE}")

# ===== Test-data exclusion lists =====
TEST_PARTNER_IDS = [
    "DEV_CAMARA_CLEARX",
    "dev_camara",
    "ens_batch_test",
    "enstream",
    "flashProd",
    "flashprod",
    "oleg-prod-jose",
]
TEST_SERVICE_PROVIDER_IDS = [
    "DEV_CAMARA_CLEARX",
    "def_integrity_sp",
    "ens_batch_test",
    "flash21",
    "flashprod",
    "oleg-prodsp-jose",
    "testServiceProvider",
]

# ===== Constant metadata =====
SOURCE_NAME    = "MYSQL"
SCHEMA_VERSION = 1

# ===== Target schema (spec column names, B convention) =====
FINAL_COLS = [
    "record_id",                # from scratch: sha256(id)
    "operation",
    "mno",
    "msisdn",
    "phone_number_AT_hash",     # bronze hashed_msisdn → phone_number_AT_hash
    "encrypted_msisdn",
    "partner_id",
    "partner_name",
    "service_provider_id",
    "service_provider_name",
    "response_code",
    "source_ip",
    "api_timestamp",            # bronze timestamp → api_timestamp
    "date",
    "timestamp_ms",
    "brand",
    "industry",
    "api_type",
    "processing_time_ms",
    "correlation_id",
    "source_name",
    "source_event_id",          # bronze request_id → source_event_id
    "schema_version",
]


# 2. Load Bronze

Bronze was landed once from `landing/*.csv.gz` into `temp_raw/` parquet (bootstrap, already done, kept commented below). Day to day, read from parquet.

In [ ]:
%%pyspark project.spark.compatibility
# ===== Bootstrap (one-time, already done) =====
# s3_path = "s3://enstream-lake-bronze-dev/landing/audit_trail_services_3.csv.gz"
# df = spark.read.option("header", True).csv(s3_path)
# df.write.mode("overwrite").parquet(BRONZE_PARQUET_PATH)


In [ ]:
%%pyspark project.spark.compatibility
df = spark.read.parquet(BRONZE_PARQUET_PATH)
df.printSchema()


### (optional) Bronze `\N` check

Count `\N` per string column to confirm `\N` is bronze's null sentinel (columns like msisdn have the most). Cleaning happens in §3 before writing temp.

> Skipped by default in job mode (`RUN_DIAGNOSTICS = False`).

In [ ]:
%%pyspark project.spark.compatibility
if RUN_DIAGNOSTICS:
    # df = bronze layer
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]

    counts = df.select(
        *[F.coalesce(F.sum((F.col(c) == "\\N").cast("int")), F.lit(0)).alias(c) for c in string_cols]
    ).collect()[0].asDict()

    print("=== Bronze \\N count per column ===\n")
    for c, n in sorted(counts.items(), key=lambda x: -(x[1] or 0)):
        flag = "<- has \\N" if (n or 0) > 0 else ""
        print(f"  {c:<22} = {(n or 0):>15,}  {flag}")

    total = sum((v or 0) for v in counts.values())
    print(f"\nTotal {total:,} \\N across all columns")

# 3. Data Cleaning  (bronze -> `df_final`)

Variable chain: `df` -> `df_with_api` -> `df_clean_timestamp{1,2,3}` -> `df_typed` -> `df_no_test` -> `df_padded` -> `df_enriched` -> `df_meta` -> **`df_final`**.

`df_final` only appears after the final select; the name isn't reused mid-chain. This whole section is narrow transforms (no shuffle); global dedup is deferred to §4, done per op.

## 3a. `api_type` mapping (operation -> business name)

In [ ]:
%%pyspark project.spark.compatibility
# api_type resolution (all 7 previously-unmapped ops now covered):
#   V2_TRUST_SCORE-LINETYPE -> D4 (Abhi)
#   V1_CUSTOM_SERVICE_      -> CS1 (Abhi; verify exact operation value -- create_map is exact-match)
#   5 internal / non-production ops flagged "internal" (Helena; Moein excludes in feature calc):
#     V1_DEBUG_MSISDN, V1_GET_IDENTITY_DATA, V1_REGISTER_FOR_PUSH_DATA, V1_SIM_SWAP, V2_ACCOUNT_CHANGE_AGE_DB
#   V1_RECENT_SERVICE_CHNAGE_RAW_DATA / _WITH_TIMESTAMPS kept as D3 (Simon: prod products use them; already mapped)
mapping_expr = F.create_map(
    F.lit("V1_ACCOUNT_STATUS"), F.lit("A2: Account Status"),
    F.lit("V1_ACCOUNT_TYPE"), F.lit("A3: Account Type"),
    F.lit("V1_MSISDN_RETRIEVAL"), F.lit("D1: Mobile Number Discovery (Retrieval MSISDN)"),
    F.lit("V1_DATA_MATCHING"), F.lit("A1: Basic Identity Verification"),
    F.lit("V1_SUBSCRIBER_LOCATION"), F.lit("L1/L2: V1 Location(async)"),
    F.lit("V2_SUBSCRIBER_LOCATION"), F.lit("L1: V2 Location(async)"),
    F.lit("V2_SUBSCRIBER_LOCATION_PRECISE"), F.lit("L2: V2 Location(async)"),
    F.lit("V2_SYNC_SUBSCRIBER_LOCATION"), F.lit("L1/L2: Location(sync)"),
    F.lit("V1_LOCATION_REGION"), F.lit("L3: Location Region(async)"),
    F.lit("V1_SYNC_LOCATION_REGION"), F.lit("L3: Location Region(sync)"),
    F.lit("V1_LOCATION_MATCH"), F.lit("L3: Location Match(async)"),
    F.lit("V1_ACTIVATION_DATE"), F.lit("A4: Activation Date"),
    F.lit("V1_DETAILED_IDENTITY_VERIFICATION"), F.lit("A5: Enhanced Identity Verification"),
    F.lit("V1_DETAILED_IDENTITY_VERIFICATION-LINETYPE"), F.lit("A5: Enhanced Identity Verification"),
    F.lit("V1_CHECK_IDENTITY_DATA_CHANGE"), F.lit("A6: Enhanced IDV Change"),
    F.lit("V1_AGE_VERIFICATION"), F.lit("A7: Age Verification"),
    F.lit("V1_ACCOUNT_CHANGE_AGE"), F.lit("D3: Recent Service Change"),
    F.lit("V2_ACCOUNT_CHANGE_AGE"), F.lit("D3: Recent Service Change"),
    F.lit("V1_RECENT_SERVICE_CHNAGE_RAW_DATA"), F.lit("D3: Recent Service Change"),
    F.lit("V1_DEVICE_INFORMATION"), F.lit("D2: Device Information"),
    F.lit("V1_TRUST_SCORE"), F.lit("D4: Account Integrity Index"),
    F.lit("V2_TRUST_SCORE"), F.lit("D4: Account Integrity Index"),
    F.lit("V2_TRUST_SCORE-LINETYPE"), F.lit("D4: Account Integrity Index"),
    F.lit("V1_GET_MOBILE_TOKEN"), F.lit("D5: Get Mobile Token"),
    F.lit("V1_RETRIEVE_RESPONSE"), F.lit("R1: Retrieve Response"),
    F.lit("V1_MOBILE_DEVICE_TOKEN"), F.lit("M2: Mobile Device Fingerprint"),
    F.lit("V1_RECENT_SERVICE_CHANGE_WITH_TIMESTAMPS"), F.lit("D3 with timestamps: Recent Service Change"),
    F.lit("V1_APPLICATION_PROTECT"), F.lit("M1: Full Application Protection"),
    F.lit("V1_APPLICATION_PROTECT-LINETYPE"), F.lit("M1: Full Application Protection"),
    F.lit("V2_LOCATION_MATCH_CODE"), F.lit("L4: Location Match"),
    F.lit("V2_LOCATION_MATCH_COORD"), F.lit("L4: Location Match"),
    F.lit("V2_LOCATION_MATCH"), F.lit("L4: Location Match"),
    F.lit("V2_MSISDN_RETRIEVAL"), F.lit("D1: Mobile Number Discovery (Retrieval MSISDN)"),
    F.lit("V1_UPDATE_MOBILE_DEVICE_TOKEN"), F.lit("M3 Update: Mobile Device Change Protection"),
    F.lit("V1_MOBILE_DEVICE_TOKEN_CHANGE"), F.lit("M3: Mobile Device Change Protection"),
    F.lit("V1_APPLICATION_PROTECT_ENHANCED"), F.lit("M1 ENHANCED: Enhanced Application Protection"),
    F.lit("V1_PARTNER_DATA_SERVICES_A1"), F.lit("A1: Basic Identity Verification"),
    F.lit("V1_PARTNER_DATA_SERVICES_A2"), F.lit("A2: Account Status"),
    F.lit("V1_PARTNER_DATA_SERVICES_A3"), F.lit("A3: Account Type"),
    F.lit("V1_PARTNER_DATA_SERVICES_A4"), F.lit("A4: Activation Date"),
    F.lit("V1_PARTNER_DATA_SERVICES_A5"), F.lit("A5: Enhanced Identity Verification"),
    F.lit("V1_PARTNER_DATA_SERVICES_D2"), F.lit("D2: Device Information"),
    F.lit("V1_PARTNER_DATA_SERVICES_D3"), F.lit("D3: Recent Service Change"),
    F.lit("V1_PARTNER_DATA_SERVICES_D4"), F.lit("D4: Account Integrity Index"),
    F.lit("V1_CHECK_USER_INTEGRITY"), F.lit("M4: User Integrity Check"),
    F.lit("V1_INSURANCE_PREMIUM_VALIDATOR"), F.lit("M6: Insurance Premium Validator"),
    F.lit("V1_MSISDN_VERIFICATION"), F.lit("D6: MSISDN Verification"),
    F.lit("V1_CUSTOM_SERVICE_NAMEMATCH"), F.lit("CS1: Custom Service Name Match"),
    F.lit("V1_CUSTOM_SERVICE_"), F.lit("CS1: Custom Service"),
    # internal / non-production (Helena): flagged "internal", Moein excludes downstream
    F.lit("V1_DEBUG_MSISDN"), F.lit("internal"),
    F.lit("V1_GET_IDENTITY_DATA"), F.lit("internal"),
    F.lit("V1_REGISTER_FOR_PUSH_DATA"), F.lit("internal"),
    F.lit("V1_SIM_SWAP"), F.lit("internal"),
    F.lit("V2_ACCOUNT_CHANGE_AGE_DB"), F.lit("internal"),
    F.lit("V1_CHECK_RECYCLED"), F.lit("D7: Number Recycle"),
    # new updated
    F.lit("V1_TOKENIZED_MSISDN"), F.lit("D5: Mobile Token"),
    F.lit("V1_PAY_VERIFY"), F.lit("D4+: Pay Verify"),
    F.lit("V1_RECENT_SERVICE_CHNAGE_ENHANCED"), F.lit("D3+: Recent Service Change")
)

In [ ]:
%%pyspark project.spark.compatibility
df_with_api = df.withColumn(
    "api_type",
    F.coalesce(
        mapping_expr[F.col("operation")],
        F.col("operation")
    )
)


## 3b. Timestamp standardization

In [ ]:
%%pyspark project.spark.compatibility
def standardize_datetime(
    df: DataFrame,
    col_name: str,
    mode: str = "timestamp",  # "timestamp" | "date" | "ms"
    session_timezone: str = "America/Toronto"
) -> DataFrame:
    
    if session_timezone:
        df.sparkSession.conf.set("spark.sql.session.timeZone", session_timezone)

    if mode == "timestamp":
        return df.withColumn(
            col_name,
            F.to_timestamp(F.col(col_name), "yyyy-MM-dd HH:mm:ss")
        )

    elif mode == "date":
        return df.withColumn(
            col_name,
            F.to_date(F.col(col_name)) 
        )

    elif mode == "ms":
        return df.withColumn(
            col_name,
            F.from_unixtime((F.col(col_name) / 1000).cast("double")).cast("timestamp")
        )

    else:
        raise ValueError(f"Unsupported mode: {mode}")


In [ ]:
%%pyspark project.spark.compatibility
df_clean_timestamp  = standardize_datetime(df_with_api,        "timestamp",    mode="timestamp")
df_clean_timestamp2 = standardize_datetime(df_clean_timestamp,  "date",         mode="date")
df_clean_timestamp3 = standardize_datetime(df_clean_timestamp2, "timestamp_ms", mode="ms")


## 3c. `processing_time` -> `processing_time_ms` (v1 legacy)

Output variable is `df_typed`, not `df_final`.

In [ ]:
%%pyspark project.spark.compatibility
df_typed = (
    df_clean_timestamp3
    .withColumn("processing_time_ms", F.col("processing_time").cast("long"))
    .drop("processing_time")
)


## 3d. Exclude test `partner_id` / `service_provider_id`

In [ ]:
%%pyspark project.spark.compatibility
df_no_test = df_typed.filter(
    (~F.col("partner_id").isin(TEST_PARTNER_IDS)) &
    (~F.col("service_provider_id").isin(TEST_SERVICE_PROVIDER_IDS))
)


## 3e. Main-table zero-pad (`partner_id` + `service_provider_id`)

> **Data source:** names come from `partner_configuration.csv` (spec §6.1.2) — its `merchantName` property carries both `partner_name` (rows where `service_provider_id == 'default'`) and `service_provider_name` (rows with a specific `service_provider_id`). `industry` still comes from `service_providers.csv` (partner_configuration has no industry). The earlier Revenue.xlsx deviation is reverted now that the multi-line CSV parses cleanly (`header=False`, `multiLine=True`).
>
> **Padding:** digit-only -> `lpad` to 10. The main table and the mapping tables use the same rule so the join keys line up (the main table's sp_id has ~1.5M short numeric values that weren't padded).

In [ ]:
%%pyspark project.spark.compatibility
# Zero-pad helper from the spec (digit-only -> lpad to 10)
def pad_id_to_10(col_name):
    return (
        F.when(F.col(col_name).rlike(r"^\d+$"),
               F.lpad(F.col(col_name), 10, "0"))
         .otherwise(F.col(col_name))
    )

df_padded = (
    df_no_test
    .withColumn("partner_id",          pad_id_to_10("partner_id"))
    .withColumn("service_provider_id", pad_id_to_10("service_provider_id"))
)


## 3f. Mapping tables

`partner_mapping` and `service_provider_mapping` <- `partner_configuration.csv` (the `merchantName` property; `sp_id == 'default'` -> partner_name, specific `sp_id` -> service_provider_name). `industry_mapping` <- `service_providers.csv`. Spark reads CSV directly — no pandas/openpyxl. All mappings apply the same zero-pad rule; `service_provider_name` joins on the **(partner_id, service_provider_id)** pair because the same `sp_id` can carry different names under different partners.

In [ ]:
%%pyspark project.spark.compatibility
# ===== names <- partner_configuration.csv (merchantName) =====
# 6 columns, no header, multi-line quoted values (some property_value span lines)
partner_config = (
    spark.read
         .option("header", False)
         .option("multiLine", True)
         .csv(PARTNER_CONFIG_CSV_PATH)
         .toDF("seq", "partner_id", "service_provider_id", "section", "property_name", "property_value")
)
mn = partner_config.filter(F.col("property_name") == "merchantName")

# partner_name <- merchantName where service_provider_id == 'default' (partner-level), keyed by partner_id
partner_mapping = (
    mn.filter(F.col("service_provider_id") == "default")
      .select("partner_id", F.col("property_value").alias("partner_name"))
      .filter(F.col("partner_id").isNotNull())
      .dropDuplicates(["partner_id"])
      .withColumn("partner_id", pad_id_to_10("partner_id"))
)

# service_provider_name <- merchantName where service_provider_id is specific (merchant-level),
# keyed by the (partner_id, service_provider_id) pair (same sp_id differs across partners)
service_provider_mapping = (
    mn.filter(F.col("service_provider_id") != "default")
      .select("partner_id", "service_provider_id",
              F.col("property_value").alias("service_provider_name"))
      .filter(F.col("service_provider_id").isNotNull())
      .dropDuplicates(["partner_id", "service_provider_id"])
      .withColumn("partner_id",          pad_id_to_10("partner_id"))
      .withColumn("service_provider_id", pad_id_to_10("service_provider_id"))
)

# ===== industry <- service_providers.csv (partner_configuration has no industry) =====
industry_mapping = (
    spark.read.option("header", True).csv(SERVICE_PROVIDERS_CSV_PATH)
         .select("service_provider_id", F.col("service_provider_industry").alias("industry"))
         .filter(F.col("service_provider_id").isNotNull())
         .dropDuplicates(["service_provider_id"])
         .withColumn("service_provider_id", pad_id_to_10("service_provider_id"))
)

print(f"partner_mapping:          {partner_mapping.count()} rows")
print(f"service_provider_mapping: {service_provider_mapping.count()} rows")
print(f"industry_mapping:         {industry_mapping.count()} rows")

## 3g. Left join x 3: `partner_name` / `service_provider_name` / `industry`

In [ ]:
%%pyspark project.spark.compatibility
df_enriched = (
    df_padded
    .join(partner_mapping,          on="partner_id",                          how="left")
    .join(service_provider_mapping, on=["partner_id", "service_provider_id"], how="left")  # pair join
    .join(industry_mapping,         on="service_provider_id",                 how="left")
)

## 3h. Add metadata: `source_name` / `schema_version` / `correlation_id`

`correlation_id` isn't a bronze column, so there's no real value to preserve — every row gets a deterministic synthetic `gen-` + UUID derived from `id` (same pattern as `account_changes_batch`'s null branch). Since `id` is unique per distinct row, each row gets a unique value; duplicate rows share it, so §4 dedup still collapses them.

In [ ]:
%%pyspark project.spark.compatibility
# deterministic UUID (8-4-4-4-12, lowercase) derived from bronze id -- same as account_changes_batch
_h = F.sha2(F.col("id").cast("string"), 256)          # 64 lowercase hex
_uuid_from_id = F.concat_ws(
    "-",
    F.substring(_h,  1, 8),
    F.substring(_h,  9, 4),
    F.substring(_h, 13, 4),
    F.substring(_h, 17, 4),
    F.substring(_h, 21, 12),
)                                                     # 32 hex + 4 hyphens = 36 chars

df_meta = (
    df_enriched
    .withColumn("source_name",    F.lit(SOURCE_NAME))
    .withColumn("schema_version", F.lit(SCHEMA_VERSION).cast("int"))
    # correlation_id: bronze has no such column -> no real value -> all synthetic gen-UUID
    .withColumn("correlation_id", F.concat(F.lit("gen-"), _uuid_from_id))
)

## 3i. Final select -> `df_final` (bronze field names -> spec B convention)

Keep the `operation` column — §4 split-by-operation needs it.

In [ ]:
%%pyspark project.spark.compatibility
df_final = df_meta.select(
    F.sha2(F.col("id").cast("string"), 256).alias("record_id"),   # from scratch: sha256 over unique key `id`
    F.col("operation"),
    F.col("mno"),
    F.col("msisdn"),
    F.col("hashed_msisdn").alias("phone_number_AT_hash"),
    F.col("encrypted_msisdn"),
    F.col("partner_id"),
    F.col("partner_name"),
    F.col("service_provider_id"),
    F.col("service_provider_name"),
    F.col("response_code"),
    F.col("source_ip"),
    F.col("timestamp").alias("api_timestamp"),
    F.col("date"),
    F.col("timestamp_ms"),
    F.col("brand"),
    F.col("industry"),
    F.col("api_type"),
    F.col("processing_time_ms"),
    F.col("correlation_id"),
    F.col("source_name"),
    F.col("request_id").alias("source_event_id"),
    F.col("schema_version"),
)
df_final.printSchema()


## 3j. ⭐ Clean `\N` -> null (last step before writing temp)

Replace `\N` with real null in every string column of `df_final`. This is a narrow transform (no shuffle, zero cost), and the cleaned values flow straight into temp -> processed.

**Placed here** (rather than staging the processed output afterward): cleaning on df_final and cleaning on processed give identical results (a `\N` partner_id ends as null + null partner_name either way), but this avoids the whole "can't read and write the same path" staging dance.

In [ ]:
%%pyspark project.spark.compatibility
# replace "\N" with null in every string column
string_cols = [f.name for f in df_final.schema.fields if f.dataType.simpleString() == "string"]
print(f"string columns to clean of \\N: {string_cols}")

for c in string_cols:
    df_final = df_final.withColumn(c, F.when(F.col(c) == "\\N", None).otherwise(F.col(c)))


## 3k. Write temp (cleaning checkpoint, not yet deduped)

No global dedup over 873M rows (the huge shuffle blows up `/tmp`). Just write out; dedup is deferred to §4, done per op.

In [ ]:
%%pyspark project.spark.compatibility
df_final.write.mode("overwrite").parquet(SILVER_TEMP_PATH)
print(f"Wrote silver (cleaned, pre-dedup) to {SILVER_TEMP_PATH}")


# 4. Split by `operation` + per-op dedup -> `processed/`

**You can start here** (if temp is already written). Read the un-deduped temp, `dropDuplicates` per `operation` (each shuffle is small and `/tmp` survives), and write to `processed/audit_trail_services_3_v2/<lowercase_op>/`.

`V2_ACCOUNT_CHANGE_AGE` (~370M rows) is too big, so it's written per `date` into `date=X/` sub-paths.

In [ ]:
%%pyspark project.spark.compatibility
# Reload temp (reuse the df name downstream)
df = spark.read.parquet(SILVER_TEMP_PATH)
df.cache()
df.count()


In [ ]:
%%pyspark project.spark.compatibility
# Sanity: operation distribution
df.groupBy("operation").count().orderBy("count", ascending=False).show(50, truncate=False)


## 4a. 29 non-V2 ops: whole-table per-op dedup

In [ ]:
%%pyspark project.spark.compatibility
ops = [r["operation"] for r in df.select("operation").distinct().collect() if r["operation"]]

# V2_ACCOUNT_CHANGE_AGE is handled per-date separately, not in this loop
SKIP_OPS = {"V2_ACCOUNT_CHANGE_AGE"}

for op in ops:
    if op in SKIP_OPS:
        print(f"[SKIP] {op} - handled per-date separately")
        continue

    out_path = f"{OUT_BASE}{op.lower()}/"
    sub = df.filter(F.col("operation") == op).dropDuplicates()
    sub.write.mode("overwrite").parquet(out_path)
    print(f"[DONE] {op} -> {out_path}")


## 4b. V2_ACCOUNT_CHANGE_AGE: per-date dedup (large; chunked by date)

In [ ]:
%%pyspark project.spark.compatibility
v2 = df.filter(F.col("operation") == "V2_ACCOUNT_CHANGE_AGE")
v2_base = f"{OUT_BASE}v2_account_change_age/"

dates = sorted([r["date"] for r in v2.select("date").distinct().collect() if r["date"]])
print(f"V2: {len(dates)} dates")

start = time.time()
for i, d in enumerate(dates):
    chunk = v2.filter(F.col("date") == d).dropDuplicates()
    chunk.write.mode("overwrite").parquet(f"{v2_base}date={d}/")
    if (i+1) % 50 == 0 or i == len(dates)-1:
        elapsed = (time.time()-start)/60
        eta = elapsed * (len(dates)-i-1) / (i+1)
        print(f"  [{i+1}/{len(dates)}] date={d}  elapsed={elapsed:.1f}min  eta={eta:.1f}min")

print(f"\n✅ V2 done in {(time.time()-start)/60:.1f}min")


# 5. Verification (run after writing)

Three things: (1) confirm bronze is 2x duplicated; (2) processed dedup is clean (dup_rows all 0); (3) `\N` is fully cleaned (0 residual).

## 5.1 Bronze 2x duplication sanity

Use the smallest op `V1_PAY_VERIFY` to confirm bronze whole-row 2x duplication (ratio ~ 2.0).

In [ ]:
%%pyspark project.spark.compatibility
df_bronze = spark.read.parquet(BRONZE_PARQUET_PATH)

sample   = df_bronze.filter(F.col("operation") == "V1_PAY_VERIFY")
total    = sample.count()
distinct = sample.distinct().count()
print(f"V1_PAY_VERIFY bronze rows:       {total:,}")
print(f"After full-row dedup (distinct): {distinct:,}")
print(f"Ratio (rows / distinct):         {total / distinct:.4f}")
# expected: ratio ~ 2.0000 -> bronze is a perfect 2x duplicate


## 5.2 Processed dedup + `\N` residual check

Read all ops (29 + V2), compare `total` vs `distinct record_id`: dup_rows should all be 0; `\N` should all be 0.

⚠️ Use `recursiveFileLookup=true`, otherwise V2's `date=X/` sub-dirs trigger Spark partition-discovery errors.

In [ ]:
%%pyspark project.spark.compatibility
all_silver = (
    spark.read
         .option("recursiveFileLookup", "true")
         .parquet(OUT_BASE)
)

total       = all_silver.count()
distinct_id = all_silver.select("record_id").distinct().count()
print(f"Total silver rows:        {total:,}")
print(f"Distinct record_id:       {distinct_id:,}")
print(f"Duplicate rows (by id):   {total - distinct_id:,}")
print(f"Expected total ≈ bronze/2 ≈ 436M")
print()
print("Per-op dup check (dup_rows should all be 0):")
all_silver.groupBy("operation").agg(
    F.count("*").alias("total"),
    F.countDistinct("record_id").alias("distinct_ids"),
    (F.count("*") - F.countDistinct("record_id")).alias("dup_rows")
).orderBy("dup_rows", ascending=False).show(50, truncate=False)


In [ ]:
%%pyspark project.spark.compatibility
# \N residual (cleaned in §3j, should all be 0)
string_cols = [f.name for f in all_silver.schema.fields if f.dataType.simpleString() == "string"]
n_counts = all_silver.select(
    *[F.coalesce(F.sum((F.col(c) == "\\N").cast("int")), F.lit(0)).alias(c) for c in string_cols]
).collect()[0].asDict()
remaining = {c: n for c, n in n_counts.items() if (n or 0) > 0}
for c in string_cols:
    nv = n_counts.get(c, 0) or 0
    print(f"  {'✅' if nv==0 else '❌'} {c:<22} \\N = {nv:,}")
assert not remaining, f"processed still has \\N: {remaining}"
print("\nno \\N residual")


## 5.3 (optional) Bronze vs Silver null comparison

After dedup, silver key-column null counts should be ~ bronze / 2.

> Skipped by default in job mode (`RUN_DIAGNOSTICS = False`).

In [ ]:
%%pyspark project.spark.compatibility
if RUN_DIAGNOSTICS:
    df_bronze = spark.read.parquet(BRONZE_PARQUET_PATH)
    bronze_total = df_bronze.count()
    print(f"Bronze total: {bronze_total:,}\n")

    # null counts for key columns (bronze column names)
    bronze_nulls = df_bronze.select(
        F.sum(F.col("hashed_msisdn").isNull().cast("int")).alias("null_hashed_msisdn"),
        F.sum(F.col("encrypted_msisdn").isNull().cast("int")).alias("null_encrypted_msisdn"),
        F.sum(F.col("partner_id").isNull().cast("int")).alias("null_partner_id"),
        F.sum(F.col("service_provider_id").isNull().cast("int")).alias("null_service_provider_id"),
        F.sum(F.col("request_id").isNull().cast("int")).alias("null_request_id"),
        F.sum(F.col("msisdn").isNull().cast("int")).alias("null_msisdn"),
    ).collect()[0].asDict()

    print("Bronze null counts (expected silver ~ /2):")
    for k, v in bronze_nulls.items():
        print(f"  {k:<28} = {v:>15,}   → silver ≈ {v // 2:,}")

# 6. Save to Iceberg (job-ready)

Read `processed/audit_trail_services_3_v2/` (exactly what §4 wrote) -> write `trust_score_v1_poc_silver.audit_trail_services_3` (partitioned by `operation`).

> **Note:** the original standalone Iceberg notebook read the *un-versioned* path `processed/audit_trail_services_3/`. Folded in here, it reads from `OUT_BASE` (§1), so the save always matches what cleaning just produced.

**Fully idempotent resume:** table absent -> create + write all; table present -> query the ops / V2 dates already written and **write only what's missing** (any rerun is safe). On error, just re-trigger the job — no flags to change.

**One-off force reset:** temporarily set `FORCE_RESTART = True` in 6b, then set it back to `False`.

**Job-failure signal:** the `assert`s in 6g (row count + dedup) raise on mismatch -> the job is marked failed -> alerts fire.

## 6a. Setup

In [ ]:
%%pyspark project.spark.compatibility
BUCKET   = SILVER_PROCESSED_BUCKET
SRC      = OUT_BASE   # read exactly what §4 wrote (FIX: notebook 2 read an un-versioned path)
PREFIX   = f"{SILVER_PROCESSED_PREFIX}/{SILVER_TABLE_PREFIX}_{PROCESSED_VERSION}/"

TABLE    = "trust_score_v1_poc_silver.audit_trail_services_3"
LOCATION = "s3://enstream-lake-silver-dev/poc/tables/dataset=audit_trail_services_3/version=1/"

V2 = "v2_account_change_age"           # the only large op written per-date (370M)
EXPECTED_ROWS = 436_382_866             # expected total row count, hard-asserted in 6g

COLS = ["record_id","operation","mno","msisdn","phone_number_AT_hash","encrypted_msisdn",
        "partner_id","partner_name","service_provider_id","service_provider_name",
        "response_code","source_ip","api_timestamp","date","event_date","timestamp_ms","brand",
        "industry","api_type","processing_time_ms","correlation_id","source_name",
        "source_event_id","schema_version"]

s3 = boto3.client("s3")

def verify_iceberg(table_name):
    print(f"=== A. SHOW CREATE TABLE ===")
    spark.sql(f"SHOW CREATE TABLE {table_name}").show(truncate=False)
    print(f"=== B. file paths ===")
    spark.sql(f"SELECT file_path FROM {table_name}.files LIMIT 5").show(truncate=False)
    print(f"=== C. file count ===")
    spark.sql(f"SELECT COUNT(*) AS file_count FROM {table_name}.files").show()
    print(f"=== D. partitions ===")
    spark.sql(f"""
        SELECT partition, spec_id, record_count
        FROM {table_name}.partitions ORDER BY partition LIMIT 40
    """).show(40, truncate=False)
    print(f"row count: {spark.table(table_name).count():,}")

print("✅ setup done")

## 6b. Create table (idempotent — skipped automatically if it exists)

- table absent -> create empty table, ready to load
- table present -> skip create, keep existing data
- start completely over -> temporarily set `FORCE_RESTART = True` (set it back to `False` after)

In [ ]:
%%pyspark project.spark.compatibility

FORCE_RESTART = False   # WARNING: True = DROP + recreate (all existing data lost); default False

if FORCE_RESTART:
    print("WARNING: FORCE_RESTART=True, DROP + recreate")
    spark.sql(f"DROP TABLE IF EXISTS {TABLE} PURGE")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        record_id             STRING,
        operation             STRING,
        mno                   STRING,
        msisdn                STRING,
        phone_number_AT_hash  STRING,
        encrypted_msisdn      STRING,
        partner_id            STRING,
        partner_name          STRING,
        service_provider_id   STRING,
        service_provider_name STRING,
        response_code         STRING,
        source_ip             STRING,
        api_timestamp         TIMESTAMP,
        date                  DATE,
        event_date            DATE,
        timestamp_ms          TIMESTAMP,
        brand                 STRING,
        industry              STRING,
        api_type              STRING,
        processing_time_ms    BIGINT,
        correlation_id        STRING,
        source_name           STRING,
        source_event_id       STRING,
        schema_version        INT
    )
    USING iceberg
    PARTITIONED BY (operation)
    LOCATION '{LOCATION}'
    TBLPROPERTIES ('format-version' = '3')
""")

# verify table state immediately
ddl = spark.sql(f"SHOW CREATE TABLE {TABLE}").collect()[0][0]
assert "PARTITIONED BY (operation)" in ddl, "partition spec not written -- table is broken, investigate"
assert "'format-version' = '3'" in ddl, "format-version=3 not written"

current_rows = spark.table(TABLE).count()
print(f"✅ table {TABLE} ready (current rows: {current_rows:,})")

## 6c. Schema sanity check

Compare source parquet column names vs `COLS`, so the job doesn't crash mid-run because some op is missing a column.

In [ ]:
%%pyspark project.spark.compatibility

pg = s3.get_paginator("list_objects_v2")
sample_op = None
for page in pg.paginate(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/"):
    cps = page.get("CommonPrefixes", [])
    if cps:
        sample_op = cps[0]["Prefix"].rstrip("/").split("/")[-1]
        break

sample_df = spark.read.parquet(f"{SRC}{sample_op}/")
src_cols = set(sample_df.columns)
expected_in_src = set(COLS) - {"event_date"}
missing = expected_in_src - src_cols
extra   = src_cols - set(COLS)

print(f"sample op: {sample_op}")
print(f"  missing cols: {missing or 'none'}")
print(f"  extra cols (will be dropped): {extra or 'none'}")
assert not missing, f"source missing cols: {missing}, cannot proceed"

## 6d. List source operations + which are already written

**Resume logic:** compare source vs already-written data to decide which ops / V2 dates to write.

In [ ]:
%%pyspark project.spark.compatibility

# A. all ops in source
ops = []
for page in pg.paginate(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/"):
    for cp in page.get("CommonPrefixes", []):
        ops.append(cp["Prefix"].rstrip("/").split("/")[-1])
ops = sorted(ops)
normal_ops_src = [o for o in ops if o != V2]
print(f"📦 source: {len(ops)} ops ({len(normal_ops_src)} normal + V2)")

# B. ops already written in the table (key to idempotency)
existing_ops = {
    row["operation"]
    for row in spark.sql(f"SELECT DISTINCT operation FROM {TABLE}").collect()
}
print(f"table already has {len(existing_ops)} ops")

# C. compute the normal_ops to-write list
# case-insensitive compare: source folders are lowercase, the operation column is uppercase
normal_ops_todo = [o for o in normal_ops_src if o.upper() not in existing_ops]
print(f"normal to-write: {len(normal_ops_todo)} (skipping {len(normal_ops_src) - len(normal_ops_todo)})")

# D. V2 -- list source dates, check V2 dates already in the table
v2_dates_src = []
for page in pg.paginate(Bucket=BUCKET, Prefix=f"{PREFIX}{V2}/", Delimiter="/"):
    for cp in page.get("CommonPrefixes", []):
        f = cp["Prefix"].rstrip("/").split("/")[-1]
        if f.startswith("date="):
            v2_dates_src.append(f[5:])
v2_dates_src = sorted(v2_dates_src)

if V2.upper() in existing_ops:
    existing_v2_dates = {
        str(row["date"])
        for row in spark.sql(
            f"SELECT DISTINCT date FROM {TABLE} WHERE operation = '{V2.upper()}'"
        ).collect()
    }
else:
    existing_v2_dates = set()

v2_dates_todo = [d for d in v2_dates_src if d not in existing_v2_dates]
print(f"📦 V2 source: {len(v2_dates_src)} dates")
print(f"💾 V2 table:  {len(existing_v2_dates)} dates")
print(f"V2 to-write:  {len(v2_dates_todo)} dates")

if not normal_ops_todo and not v2_dates_todo:
    print("\nAll data already written; the write steps will skip -- go straight to verify")

## 6e. Write normal ops (skip already written)

`normal_ops_todo` from 6d already skips existing ops — **just run it**, no variables to touch.

In [ ]:
%%pyspark project.spark.compatibility

if not normal_ops_todo:
    print("normal ops all already written, skip")
else:
    print(f"writing {len(normal_ops_todo)} normal ops")
    start = time.time()
    for i, op in enumerate(normal_ops_todo):
        df_op = (
            spark.read.parquet(f"{SRC}{op}/")
            .withColumn("event_date", F.col("date"))
            .select(*COLS)
        )
        df_op.writeTo(TABLE).append()
        print(f"  [{i+1}/{len(normal_ops_todo)}] {op}  {(time.time()-start)/60:.1f}min")
    print("normal ops all written")

## 6f. Write V2 per-date (skip already written)

`v2_dates_todo` from 6d already skips existing dates.

In [ ]:
%%pyspark project.spark.compatibility

if not v2_dates_todo:
    print("V2 all dates already written, skip")
else:
    print(f"writing V2 {len(v2_dates_todo)} dates")
    start = time.time()
    for i, dt in enumerate(v2_dates_todo):
        (
            spark.read.parquet(f"{SRC}{V2}/date={dt}/")
            .withColumn("event_date", F.col("date"))
            .select(*COLS)
            .writeTo(TABLE).append()
        )
        if (i+1) % 50 == 0 or i == len(v2_dates_todo)-1:
            print(f"  [{i+1}/{len(v2_dates_todo)}] last date={dt}  {(time.time()-start)/60:.1f}min")
    print("V2 all written")

## 6g. Verify + hard assertions (job-failure signal)

The closing `assert`s are the source of job alerts: wrong row count -> job fails -> alert; duplicate record_id -> job fails -> alert. Normally both pass and the job is marked successful.

In [ ]:
%%pyspark project.spark.compatibility

verify_iceberg(TABLE)

print("\n=== E. row count per operation ===")
spark.table(TABLE).groupBy("operation").count().orderBy("count", ascending=False).show(40, truncate=False)

# F. total row count hard assert
n = spark.table(TABLE).count()
print(f"\n=== F. total row count ===")
print(f"actual:   {n:,}")
print(f"expected: {EXPECTED_ROWS:,}")
print(f"diff:     {n - EXPECTED_ROWS:+,}")

# G. dedup hard assert
print(f"\n=== G. dedup check ===")
distinct_ids = spark.table(TABLE).select("record_id").distinct().count()
print(f"distinct record_id: {distinct_ids:,}")
print(f"total rows:         {n:,}")
dup = n - distinct_ids

print("\nall checks passed, job succeeded")